# 03 – Walk-forward backtest and metrics (tasks 7–8)

Runs `src/backtest.py` for the four portfolios and checks the result.
Timing and cost conventions: module docstring and `docs/open_decisions.md`.

In [1]:
# Setup
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "data.py").exists())
sys.path.insert(0, str(ROOT))
from src import backtest, data
from src.optimizer import BENCH, check_rules

TICKERS = data.TICKERS
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", None)

## 1. Short sample first (2016)

In [2]:
daily, monthly = backtest.run_backtest("full", first="2015-12", last="2016-11")
print(f"Sessions: {daily.index[0].date()} -> {daily.index[-1].date()} | {len(daily)}")
print(monthly[["traded", "cost", "te_ex_ante", "relaxed_turnover_cap"]].round(4))

Sessions: 2016-01-04 -> 2016-12-30 | 252
         traded    cost  te_ex_ante relaxed_turnover_cap
month                                                   
2015-12  0.2072  0.0001       0.017                 None
2016-01  0.2004  0.0001       0.020                 None
2016-02  0.2112  0.0001       0.020                 None
2016-03  0.1995  0.0001       0.020                 None
2016-04  0.1961  0.0001       0.020                 None
2016-05  0.1999  0.0001       0.020                 None
2016-06  0.1984  0.0001       0.020                 None
2016-07  0.2029  0.0001       0.020                 None
2016-08  0.1999  0.0001       0.020                 None
2016-09  0.1527  0.0001       0.020                 None
2016-10  0.1834  0.0001       0.020                 None
2016-11  0.2007  0.0001       0.020                 None


## 2. Full run: benchmark, BL with TE budget only, BL under full mandate (saved to `data/results/`)

In [3]:
net, gross, log = backtest.run_all()
print(f"Evaluation: {net.index[0].date()} -> {net.index[-1].date()} | {len(net)} sessions")
print("Files:", sorted(p.name for p in backtest.RESULTS_DIR.glob("*.csv")))

Evaluation: 2016-01-04 -> 2026-10-30 | 2723 sessions
Files: ['daily_returns_gross.csv', 'daily_returns_net.csv', 'metrics.csv', 'metrics_by_year.csv', 'monthly_log.csv']


## 3. Checks

In [4]:
W = log.set_index(["portfolio", "month"])[[f"w_{t}" for t in TICKERS]].set_axis(TICKERS, axis=1)
rules = {"benchmark": "full", "bl_te_only": "long_only", "bl_full": "full"}
viol = sum(bool(check_rules(x, rules[p])) for (p, _), x in W.iterrows())
print(f"Weights sum to 1: max error {(W.sum(axis=1) - 1).abs().max():.1e} | rule violations: {viol}")

relaxed = log.dropna(subset=["relaxed_turnover_cap"])
print("Turnover cap relaxed:", relaxed[["portfolio", "month", "relaxed_turnover_cap"]].to_dict("records"))

cost_gap = ((1 + gross).prod() / (1 + net).prod() - 1)
print("Cumulative cost drag:", cost_gap.round(5).to_dict())

# Benchmark vs static weights on monthly returns: gap = intra-month drift + next-day rebalance only
_, _, mon = data.load_processed()
static = mon.loc["2016-01":"2026-10", TICKERS] @ BENCH
realized = (1 + net["benchmark"]).groupby(net.index.to_period("M")).prod() - 1
print(f"Benchmark vs static monthly: mean |gap| {(realized - static).abs().mean():.2e}, "
      f"max {(realized - static).abs().max():.2e} ({(realized - static).abs().idxmax()})")

Weights sum to 1: max error 1.2e-15 | rule violations: 0
Turnover cap relaxed: [{'portfolio': 'bl_te_only', 'month': Period('2020-03', 'M'), 'relaxed_turnover_cap': 0.390625}, {'portfolio': 'bl_full', 'month': Period('2020-03', 'M'), 'relaxed_turnover_cap': 0.3125}]
Cumulative cost drag: {'benchmark': 0.0, 'bl_te_only': 0.01162, 'bl_full': 0.01146}
Benchmark vs static monthly: mean |gap| 1.52e-04, max 4.80e-03 (2020-04)


## 4. Metrics (task 8)

`src/metrics.py`, saved to `data/results/metrics.csv` and `metrics_by_year.csv` for the app.
Net of 5 bp costs; annualized with 252 sessions; ES as a positive daily loss.

In [5]:
from src import metrics

table, yearly = metrics.compute_all()
print(table.T.round(4))

                              benchmark  bl_te_only  bl_full
Return (CAGR)                    0.0817      0.0873   0.0845
Volatility                       0.1042      0.1079   0.1080
Sharpe                           0.5855      0.6169   0.5925
Active return (net)              0.0000      0.0056   0.0030
Active return (gross)            0.0000      0.0066   0.0040
Active CAGR                      0.0000      0.0056   0.0028
Realized TE                      0.0000      0.0236   0.0234
Information ratio                   NaN      0.2354   0.1274
t-stat active return                NaN      0.7737   0.4186
Turnover (two-way, per year)     0.2571      2.1373   2.1098
Cost (per year)                  0.0000      0.0011   0.0011
Max drawdown                     0.2415      0.2406   0.2416
Max relative drawdown            0.0000      0.0567   0.0610
ES 95% (daily)                   0.0154      0.0163   0.0163
Active ES 95% (daily)            0.0000      0.0035   0.0035


Realized TE and active return by calendar year (is the 2% budget respected out of sample?)

In [6]:
print(yearly.round(4))

     Active return         Realized TE        
        bl_te_only bl_full  bl_te_only bl_full
Date                                          
2016       -0.0298 -0.0315      0.0205  0.0212
2017        0.0305  0.0278      0.0171  0.0170
2018       -0.0192 -0.0274      0.0292  0.0285
2019       -0.0131 -0.0153      0.0175  0.0186
2020        0.0348  0.0268      0.0354  0.0351
2021        0.0027  0.0026      0.0181  0.0181
2022       -0.0050 -0.0064      0.0236  0.0235
2023        0.0044  0.0045      0.0183  0.0183
2024        0.0386  0.0297      0.0225  0.0201
2025        0.0122  0.0183      0.0221  0.0220
2026        0.0049  0.0036      0.0286  0.0285
